In [ ]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))


# Fase 3: Codificación y Preparación de Datos

En esta fase del proyecto se realiza la **transformación de variables** para preparar el dataset para modelos de *Machine Learning*.  
El objetivo es convertir todas las variables en formato **numérico**, reducir la **dimensionalidad** y asegurar que los datos estén listos para el entrenamiento de modelos.

### Objetivos de esta fase

1. Analizar variables categóricas del dataset.
2. Reducir categorías poco frecuentes (especialmente en variables como `Degree`).
3. Aplicar técnicas de codificación:
   - **OneHotEncoder** para variables categóricas.
   - **OrdinalEncoder** para variables binarias.
   - **StandardScaler** para variables numéricas.
4. Construir un **pipeline de transformación** usando `ColumnTransformer`.
5. Generar un dataset final listo para modelado.

Dataset utilizado: **Student Depression Dataset (versión limpia)**. Para saber cómo se obtuvo este dataset, consultar el notebook `Fase_2B_limpieza.ipynb`


In [ ]:
import numpy as np
import pandas as pd
from src.carga import cargar_csv
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from src.eda_utils import *

In [ ]:
df = cargar_csv("..\data\\processed\Student_Depression_Dataset_Limpio.csv")

In [ ]:
df_codificado = df.copy()
df_codificado.info()

| Método         | Qué hace                                                   |
| -------------- | -----------------------------------------------------------|
| LabelEncoder   | convierte categorías en números                            |
| OneHotEncoder  | crea columnas binarias                                     |
| OrdinalEncoder | asigna números pero pensado para variables ordinales       |
| StandardScaler | normaliza datos númericos y los asigna en una misma escala |

In [ ]:
df_codificado.select_dtypes(include='object').nunique()

## Reducción de cardinalidad

Algunas variables categóricas contienen una gran cantidad de categorías (por ejemplo, `Degree`).

Para evitar:
- Explosión de dimensiones en One-Hot Encoding
- Sobreajuste en modelos

Se agrupan las categorías menos frecuentes en una categoría general (por ejemplo: "Other").

In [ ]:
# Columnas que tienen muchas categorías y que queremos simplificar
# En este caso se seleccionó 'Degree' porque suele tener muchos valores distintos
cols = ['Degree']

# Recorremos cada columna de la lista
for col in cols:
    # 1. Contamos cuántas veces aparece cada categoría
    # value_counts() devuelve las categorías ordenadas por frecuencia
    # nlargest(10) selecciona solo las 10 más frecuentes
    top = df_codificado[col].value_counts().nlargest(10).index
    
    # 2. Reemplazamos las categorías poco frecuentes
    # Si el valor está dentro de las 10 más comunes se mantiene
    # Si no, se reemplaza por la categoría "Other"
    df_codificado[col] = df_codificado[col].apply(lambda x: x if x in top else 'Other')

In [ ]:
df_codificado['Degree'].unique()


## Construcción del Pipeline de Transformación

Para preparar los datos se utiliza **ColumnTransformer**, lo que permite aplicar diferentes transformaciones según el tipo de variable.

### Transformaciones aplicadas

**Variables categóricas**
- Se utiliza `OneHotEncoder`.
- Se aplica `drop='first'` para eliminar una categoría redundante y evitar multicolinealidad.

**Variable binaria**
- Se utiliza `OrdinalEncoder` para convertir respuestas tipo *Yes/No* en valores numéricos.

**Variables numéricas**
- Se utiliza `StandardScaler` para estandarizar los datos.
- Esto transforma las variables para que tengan:
  - Media = 0
  - Desviación estándar = 1

Esto mejora el rendimiento de varios algoritmos de Machine Learning.


In [ ]:
# 1. Eliminamos columnas innecesarias
X = df_codificado.drop(columns=['id', 'City'])

# 2. Creamos el pipeline
pipeline = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
         ['Gender', 'Sleep Duration', 'Dietary Habits', 'Degree']),

        ('bin', OrdinalEncoder(),
         ['Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),

        ('num', StandardScaler(),
         ['Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction', 'Work/Study Hours', 'Financial Stress']),
         
    ],
    remainder='passthrough' # Mantiene las columnas no mencionadas (como Depression) sin tocarlas
)

# 3. Aplicamos la transformación
x_transf = pipeline.fit_transform(X)

# 4. Reconstruimos el DataFrame
df_transf = pd.DataFrame(
    x_transf, 
    columns=pipeline.get_feature_names_out()
)
df_transf

Con este bloque podemos conocer los nombres asociados a cada columna en el pipeline

In [ ]:
for col in pipeline.get_feature_names_out():
    print(col)

## Verificación de resultados

Se revisa la dimensionalidad del dataset transformado y se valida que coincida con el número de variables generadas por el pipeline. El dataset transformado se guarda para ser utilizado en la fase de modelamiento.

In [ ]:
print(x_transf.shape)
print(len(pipeline.get_feature_names_out()))

Se logró un dataset codificado con 25 columnas y 27.817 datos

In [ ]:
df_transf.to_csv("..\data\processed\Student_Depression_Dataset_codificado.csv", index=False)